# Tutorial: Basic STREAM data access

This tutorial introduces the public `StreamClient` interface. It shows how to discover sites, read a small water-quality and streamflow subset, retrieve catchment attributes, and combine the results.

The client will request your HydroShare credentials when it is created. This notebook reads remote Parquet data without downloading the complete STREAM resource.

## 1. Connect to HydroShare

In [2]:
from stream_utils.client import StreamClient
from stream_utils.joins import join_timeseries

stream = StreamClient()

Please Enter Your HydroShare Credentials


List the dataset names recognized by the client.

In [3]:
stream.list_datasets()

['anthropogenic',
 'grab_samples',
 'land_cover',
 'metadata',
 'meteorology',
 'static_catchment_characteristics',
 'streamflow',
 'water_quality',
 'watersheds']

## 2. Find suitable monitoring sites

Find sites that provide both **water temperature** and **dissolved oxygen**. A site must contain every requested variable.

In [4]:
sites = stream.find_sites(variables=["WTemp_C", "DO_mgL"])

print(f"Found {len(sites):,} matching sites")
sites.head()

Found 1,660 matching sites


,STREAM_ID,site_name,state_name,latitude_wgs84,longitude_wgs84,WQ_parameters,streamflow
0,STREAM-gauge-1,LEAF RIVER NEAR HATTIESBURG,Mississippi,31.306583,-89.264111,"WTemp_C,DO_mgL,pH",Not available
1,STREAM-gauge-21,"RICHIES BAYOU AT SHERARD, MS",Mississippi,34.209222,-90.700194,"WTemp_C,SpC_uScm,DO_mgL,pH,Turb_FNU",Available
2,STREAM-gauge-22,"HARRIS BAYOU AT PALMER RD EAST OF ALLIGATOR, MS",Mississippi,34.074750,-90.624944,"WTemp_C,SpC_uScm,DO_mgL,pH,Turb_FNU",Available
3,STREAM-gauge-24,"BIG SUNFLOWER RIVER AT SUNFLOWER, MS",Mississippi,33.547356,-90.543158,"WTemp_C,SpC_uScm,DO_mgL,pH,Turb_FNU,NO3_mgNL",Available
4,STREAM-gauge-25,"PORTER BAYOU AT STEPHENSVILLE, MS",Mississippi,33.548250,-90.672361,"WTemp_C,SpC_uScm,DO_mgL,pH,Turb_FNU",Available


For a reproducible example, use a site known to contain these variables and **streamflow**. If that site is unavailable in a future resource version, use the first matching site instead.

In [13]:
example_site = "STREAM-gauge-22"

if sites.empty:
    raise ValueError("No sites contain all requested variables")
if example_site not in sites["STREAM_ID"].values:
    example_site = sites.iloc[0]["STREAM_ID"]

print(f"Conditioned matched, using {example_site} for the rest of the tutorial")

Conditioned matched, using STREAM-gauge-22 for the rest of the tutorial


## 3. Read water-quality observations

First, retrieve the complete water-quality record for the selected site and variables. We can then examine the earliest and latest timestamps to determine the period when observations are available.

In [22]:
water_quality = stream.read_water_quality(
    site_ids=example_site,
    variables=["WTemp_C", "DO_mgL"]
)

water_quality

,STREAM_ID,DateTime,WTemp_C,Flag_WTemp_C,DO_mgL,Flag_DO_mgL
0,STREAM-gauge-22,2022-02-09 18:00:00,7.100,A,12.000,A
1,STREAM-gauge-22,2022-02-09 19:00:00,7.525,A,11.900,A
2,STREAM-gauge-22,2022-02-09 20:00:00,7.825,A,12.000,A
3,STREAM-gauge-22,2022-02-09 21:00:00,8.025,A,12.000,A
4,STREAM-gauge-22,2022-02-09 22:00:00,8.125,A,11.950,A
...,...,...,...,...,...,...
22503,STREAM-gauge-22,2025-02-20 02:00:00,4.850,A,10.375,P
22504,STREAM-gauge-22,2025-02-20 03:00:00,4.675,A,10.400,P
22505,STREAM-gauge-22,2025-02-20 04:00:00,4.400,A,10.450,P
22506,STREAM-gauge-22,2025-02-20 05:00:00,4.200,A,10.500,P


Read one month rather than the full record. The query selects the station partition, date range, variables, and approved quality flags before converting the result to pandas.

In [24]:
water_quality = stream.read_water_quality(
    site_ids=example_site,
    variables=["WTemp_C", "DO_mgL"],
    start="2022-02-10",
    end="2022-03-10 23:59:59",
    quality="approved",
)

print(f"Retrieved {len(water_quality):,} water-quality records")
water_quality.head()

Retrieved 695 water-quality records


,STREAM_ID,DateTime,WTemp_C,Flag_WTemp_C,DO_mgL,Flag_DO_mgL
0,STREAM-gauge-22,2022-02-10 00:00:00,8.200,A,11.800,A
1,STREAM-gauge-22,2022-02-10 01:00:00,8.350,A,11.550,A
2,STREAM-gauge-22,2022-02-10 02:00:00,8.550,A,11.225,A
3,STREAM-gauge-22,2022-02-10 03:00:00,8.875,A,10.950,A
4,STREAM-gauge-22,2022-02-10 04:00:00,9.100,A,10.975,A


## 4. Read streamflow observations

Use the same station, dates, and quality policy so the two hourly datasets can be aligned.

In [32]:
streamflow = stream.read_streamflow(
    site_ids=example_site,
    start="2022-02-10",
    end="2022-03-10 23:59:59",
    quality="approved",
)

print(f"Retrieved {len(streamflow):,} streamflow records")
streamflow.head()

Retrieved 696 streamflow records


,STREAM_ID,DateTime,Q_m3s,Flag_Q_m3s
0,STREAM-gauge-22,2022-02-10 00:00:00,1.931917,A
1,STREAM-gauge-22,2022-02-10 01:00:00,2.208006,A
2,STREAM-gauge-22,2022-02-10 02:00:00,2.418259,A
3,STREAM-gauge-22,2022-02-10 03:00:00,2.525155,A
4,STREAM-gauge-22,2022-02-10 04:00:00,2.518784,A


## 5. Read static catchment attributes

Static attributes have one row per site. Here we request stream order, mean elevation, forest cover, and urban cover.

In [26]:
attributes = stream.read_static_attributes(
    site_ids=example_site,
    columns=["stream_order", "ele_mt_sav", "for_pc_sse", "urb_pc_sse"],
)

attributes

,STREAM_ID,stream_order,ele_mt_sav,for_pc_sse,urb_pc_sse
0,STREAM-gauge-22,4.0,48.687893,7.60728,0.321193


## 6. Combine the datasets

Match hourly water quality and streamflow using exact `STREAM_ID` and `DateTime` values, then attach the site-level catchment attributes. `join_timeseries()` uses pandas internally but first protects against missing keys and duplicate station/timestamp rows that could silently produce an incorrect many-to-many merge. It also gives a clear error when periods do not overlap or no timestamps match exactly.

In [35]:
combined_timeseries = join_timeseries(
    water_quality,
    streamflow,
    suffixes=("_wq", "_flow"),
)

combined = combined_timeseries.merge(
    attributes,
    on="STREAM_ID",
    how="left",
)

print(f"Combined dataset contains {len(combined):,} rows")
combined.head()

Combined dataset contains 695 rows


,STREAM_ID,DateTime,WTemp_C,Flag_WTemp_C,DO_mgL,Flag_DO_mgL,Q_m3s,Flag_Q_m3s,stream_order,ele_mt_sav,for_pc_sse,urb_pc_sse
0,STREAM-gauge-22,2022-02-10 00:00:00,8.200,A,11.800,A,1.931917,A,4.0,48.687893,7.60728,0.321193
1,STREAM-gauge-22,2022-02-10 01:00:00,8.350,A,11.550,A,2.208006,A,4.0,48.687893,7.60728,0.321193
2,STREAM-gauge-22,2022-02-10 02:00:00,8.550,A,11.225,A,2.418259,A,4.0,48.687893,7.60728,0.321193
3,STREAM-gauge-22,2022-02-10 03:00:00,8.875,A,10.950,A,2.525155,A,4.0,48.687893,7.60728,0.321193
4,STREAM-gauge-22,2022-02-10 04:00:00,9.100,A,10.975,A,2.518784,A,4.0,48.687893,7.60728,0.321193


## 7. Inspect and optionally export the subset

In [36]:
combined[["WTemp_C", "DO_mgL", "Q_m3s"]].describe()

,WTemp_C,DO_mgL,Q_m3s
count,695.000000,695.000000,695.000000
mean,10.505324,9.904089,4.502155
std,3.697053,1.431086,3.351013
min,3.300000,7.400000,1.042768
25%,7.987500,8.700000,2.063590
50%,10.600000,9.700000,3.008665
75%,13.212500,11.000000,6.013790
max,18.500000,13.200000,13.478819


In [19]:
# Uncomment to save the analysis-ready subset.
# combined.to_parquet("stream_basic_subset.parquet", index=False)

## Next steps

You can change the site filters, variables, time period, and accepted quality flags while keeping the same workflow. Later tutorials will cover spatial selection, temporal summaries, and event-based analysis.